# Export Datawrapper files

Reads `source/precinct_results.csv`, `source/census_precinct.csv`, and `source/precincts.geojson`. Writes:

- `datawrapper/precincts.csv` — one row per precinct, for the choropleth
- `datawrapper/districts.csv` — one row per supervisor district, for bar charts
- `datawrapper/precincts.geojson` — the shapes Datawrapper joins to the precinct CSV

Percentages are on a 0–100 scale, one decimal place. A rate is left blank when the precinct counted fewer than 50 ballots. `chan_pct` is also blank outside CA-11.

Open this notebook from the folder after you upload it to Drive. It does not download anything and it does not upload anything.


In [ ]:
from pathlib import Path
import json

import pandas as pd

FOLDER_NAME = "sf-2026-drive"
MIN_BALLOTS = 50


def find_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, here.parent, *list(here.parents)[:4]]
    for candidate in candidates:
        if (candidate / "source" / "precinct_results.csv").exists():
            return candidate
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        base = Path("/content/drive/MyDrive")
        direct = base / FOLDER_NAME
        if (direct / "source" / "precinct_results.csv").exists():
            return direct
        matches = [
            path for path in base.rglob("precinct_results.csv") if path.parent.name == "source"
        ]
        if len(matches) == 1:
            return matches[0].parent.parent
    except ImportError:
        pass
    raise FileNotFoundError(
        "Could not find source/precinct_results.csv. "
        "Upload this folder to My Drive and set FOLDER_NAME to the folder name."
    )


ROOT = find_root()
SOURCE = ROOT / "source"
OUT = ROOT / "datawrapper"
OUT.mkdir(parents=True, exist_ok=True)
print(ROOT)


In [ ]:
votes = pd.read_csv(SOURCE / "precinct_results.csv", dtype={"precinct_id": str})
census = pd.read_csv(SOURCE / "census_precinct.csv", dtype={"precinct_id": str})
frame = votes.merge(census, on="precinct_id", how="left")

enough = frame["ballots"].fillna(0) >= MIN_BALLOTS
in_ca11 = frame["Congressional"] == 11


def rate(numerator, denominator, keep):
    share = numerator / denominator.where(denominator > 0)
    return (share * 100).round(1).where(keep)


def as_int(column):
    return frame[column].fillna(0).round().astype(int)


precincts = pd.DataFrame({
    "precinct_id": frame["precinct_id"].astype(int),
    "precinct_name": frame["VotingPrecinctName"],
    "neighborhood": frame["Neighborhood"],
    "supervisor_district": frame["Supervisorial"].astype(int),
    "congressional_district": frame["Congressional"].astype(int),
    "assembly_district": frame["Assembly"].astype(int),
    "registered_voters": as_int("registered"),
    "ballots_counted": as_int("ballots"),
    "mail_ballots": as_int("mail_ballots"),
    "turnout_pct": rate(frame["ballots"], frame["registered"], enough),
    "mail_pct": rate(frame["mail_ballots"], frame["ballots"], enough),
    "hilton_votes": as_int("hilton"),
    "becerra_votes": as_int("becerra"),
    "hilton_pct": rate(frame["hilton"], frame["hilton"] + frame["becerra"], enough),
    "chan_votes": as_int("chan"),
    "wiener_votes": as_int("wiener"),
    "chan_pct": rate(frame["chan"], frame["chan"] + frame["wiener"], enough & in_ca11),
    "measure_b_yes": as_int("measure_b__yes"),
    "measure_b_no": as_int("measure_b__no"),
    "measure_b_yes_pct": rate(
        frame["measure_b__yes"], frame["measure_b__yes"] + frame["measure_b__no"], enough
    ),
    "measure_d_yes": as_int("measure_d__yes"),
    "measure_d_no": as_int("measure_d__no"),
    "measure_d_yes_pct": rate(
        frame["measure_d__yes"], frame["measure_d__yes"] + frame["measure_d__no"], enough
    ),
    "measure_g_yes": as_int("measure_g__yes"),
    "measure_g_no": as_int("measure_g__no"),
    "measure_g_yes_pct": rate(
        frame["measure_g__yes"], frame["measure_g__yes"] + frame["measure_g__no"], enough
    ),
    "measure_i_yes": as_int("measure_i__yes"),
    "measure_i_no": as_int("measure_i__no"),
    "measure_i_yes_pct": rate(
        frame["measure_i__yes"], frame["measure_i__yes"] + frame["measure_i__no"], enough
    ),
    "pop_2020": frame["pop"].round().astype("Int64"),
    "hispanic_pct": rate(frame["hispanic"], frame["pop"], frame["pop"] > 0),
    "nh_white_pct": rate(frame["nh_white"], frame["pop"], frame["pop"] > 0),
    "nh_black_pct": rate(frame["nh_black"], frame["pop"], frame["pop"] > 0),
    "nh_asian_pct": rate(frame["nh_asian"], frame["pop"], frame["pop"] > 0),
    "bachelors_pct": rate(frame["ba_plus"], frame["b15003_e001"], frame["b15003_e001"] > 0),
    "renter_pct": rate(frame["b25003_e003"], frame["b25003_e001"], frame["b25003_e001"] > 0),
    "hh_150k_pct": rate(frame["hh_150k"], frame["b19001_e001"], frame["b19001_e001"] > 0),
    "foreign_born_pct": rate(frame["foreign_born"], frame["b05001_e001"], frame["b05001_e001"] > 0),
    "group_quarters_pct": rate(frame["gq"], frame["pop"], frame["pop"] > 0),
})
precincts = precincts.sort_values("precinct_id")
precincts.to_csv(OUT / "precincts.csv", index=False)
print(len(precincts), "precincts ->", OUT / "precincts.csv")


In [ ]:
def weighted(part, numerator, denominator):
    if isinstance(denominator, list):
        denom = sum(part[column].fillna(0).sum() for column in denominator)
    else:
        denom = part[denominator].fillna(0).sum()
    if denom <= 0:
        return None
    return round(100 * part[numerator].fillna(0).sum() / denom, 1)


rows = []
for district, part in frame.groupby("Supervisorial"):
    ca11 = part[part["Congressional"] == 11]
    rows.append({
        "supervisor_district": int(district),
        "ballots_counted": int(part["ballots"].fillna(0).sum()),
        "registered_voters": int(part["registered"].fillna(0).sum()),
        "turnout_pct": weighted(part, "ballots", "registered"),
        "hilton_votes": int(part["hilton"].fillna(0).sum()),
        "becerra_votes": int(part["becerra"].fillna(0).sum()),
        "hilton_pct": weighted(part, "hilton", ["hilton", "becerra"]),
        "ca11_ballots": int(ca11["ballots"].fillna(0).sum()),
        "chan_votes": int(ca11["chan"].fillna(0).sum()),
        "wiener_votes": int(ca11["wiener"].fillna(0).sum()),
        "chan_pct": weighted(ca11, "chan", ["chan", "wiener"]),
        "measure_b_yes_pct": weighted(part, "measure_b__yes", ["measure_b__yes", "measure_b__no"]),
        "measure_d_yes_pct": weighted(part, "measure_d__yes", ["measure_d__yes", "measure_d__no"]),
        "measure_g_yes_pct": weighted(part, "measure_g__yes", ["measure_g__yes", "measure_g__no"]),
        "measure_i_yes_pct": weighted(part, "measure_i__yes", ["measure_i__yes", "measure_i__no"]),
    })

districts = pd.DataFrame(rows).sort_values("supervisor_district")
districts.to_csv(OUT / "districts.csv", index=False)
print(districts[["supervisor_district", "hilton_pct", "chan_pct", "measure_b_yes_pct"]].to_string(index=False))


In [ ]:
shapes = json.loads((SOURCE / "precincts.geojson").read_text())
for feature in shapes["features"]:
    props = feature["properties"]
    feature["properties"] = {
        "precinct_id": int(props["precinct_id"]),
        "precinct_name": props["VotingPrecinctName"],
        "neighborhood": props["Neighborhood"],
        "supervisor_district": int(props["Supervisorial"]),
        "congressional_district": int(props["Congressional"]),
        "assembly_district": int(props["Assembly"]),
    }
shapes["crs"] = {"type": "name", "properties": {"name": "EPSG:4326"}}
(OUT / "precincts.geojson").write_text(json.dumps(shapes))
print(len(shapes["features"]), "shapes ->", OUT / "precincts.geojson")


In [ ]:
hilton = precincts["hilton_votes"].sum()
becerra = precincts["becerra_votes"].sum()
print("Hilton share", round(100 * hilton / (hilton + becerra), 2))
print("precincts", len(precincts), "with hilton_pct", precincts["hilton_pct"].notna().sum())
print("chan_pct filled", precincts["chan_pct"].notna().sum())
print("blank chan outside CA-11", int((precincts["congressional_district"] != 11).sum()))


## Load these into Datawrapper

For a precinct choropleth, upload `datawrapper/precincts.geojson` as the map and `datawrapper/precincts.csv` as the data. Join on `precinct_id`. Color one of `turnout_pct`, `mail_pct`, `hilton_pct`, `chan_pct`, `measure_b_yes_pct`, `measure_d_yes_pct`, `measure_g_yes_pct`, or `measure_i_yes_pct`.

For a supervisor-district bar chart, upload `datawrapper/districts.csv`. One row per district.
